# Apollo 13 — staged, resumable headless render
Run cells in order. FINAL is manually gated and never starts automatically.


In [ ]:
# 1. Runtime and GPU setup
import os, subprocess, pathlib
PROJECT = pathlib.Path('/content/apollo13_blender')
print(subprocess.getoutput('nvidia-smi || true'))
USE_DRIVE = False
if USE_DRIVE:
 from google.colab import drive; drive.mount('/content/drive')
 WORKROOT = pathlib.Path('/content/drive/MyDrive/apollo13_render') if USE_DRIVE else pathlib.Path('/content')
 WORKROOT.mkdir(parents=True, exist_ok=True)


In [ ]:
# 2. Install Blender and FFmpeg (pin/checksum may be updated after verifying Blender release archive)
!apt-get -qq update && apt-get -qq install -y ffmpeg
!wget -q https://download.blender.org/release/Blender4.2/blender-4.2.0-linux-x64.tar.xz -O /content/blender.tar.xz
!tar -xf /content/blender.tar.xz -C /content
BLENDER='/content/blender-4.2.0-linux-x64/blender'
!$BLENDER --version
!ffmpeg -version | head -1


In [ ]:
# 3. GitHub repository dialog — paste any public GitHub HTTPS clone URL, then click Clone / update.
# Run this cell once, fill in the visible fields, click the button, and wait for
# "Ready" before proceeding to cell 4.
import json, shutil, subprocess
from IPython.display import display, clear_output
import ipywidgets as widgets

repo_url_box = widgets.Text(
    value="",
    placeholder="https://github.com/OWNER/REPOSITORY.git",
    description="GitHub URL:",
    layout=widgets.Layout(width="720px"),
)
branch_box = widgets.Text(
    value="main", placeholder="main", description="Branch:",
    layout=widgets.Layout(width="360px"),
)
clone_button = widgets.Button(description="Clone / update repository", button_style="primary", icon="download")
clone_status = widgets.Output(layout=widgets.Layout(border="1px solid #444", padding="10px"))
display(widgets.VBox([
    widgets.HTML("<b>Repository setup</b><br>Paste your GitHub HTTPS clone URL and choose its branch."),
    repo_url_box, branch_box, clone_button, clone_status,
]))

def clone_selected_repository(_):
    """Clone/update safely and locate the Blender project without shell interpolation."""
    global PROJECT, REPO_URL, REPO_BRANCH
    with clone_status:
        clear_output()
        REPO_URL = repo_url_box.value.strip()
        REPO_BRANCH = branch_box.value.strip() or "main"
        if not REPO_URL.startswith(("https://github.com/", "git@github.com:")):
            print("Enter a GitHub HTTPS URL, e.g. https://github.com/account/repository.git")
            return
        clone_dir = WORKROOT / "apollo13_source"
        try:
            if (clone_dir / ".git").exists():
                subprocess.run(["git", "-C", str(clone_dir), "fetch", "--depth", "1", "origin", REPO_BRANCH], check=True)
                subprocess.run(["git", "-C", str(clone_dir), "checkout", "-B", REPO_BRANCH, f"origin/{REPO_BRANCH}"], check=True)
            else:
                subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(clone_dir)], check=True)
        except subprocess.CalledProcessError as error:
            print(f"Clone failed (exit {error.returncode}). Check the URL, branch, and repository access.")
            return
        candidates = [clone_dir, clone_dir / "apollo13_blender"]
        PROJECT = next((item for item in candidates if (item / "production_manifest.json").is_file()), None)
        if PROJECT is None:
            print("Clone succeeded, but no production_manifest.json was found at repo root or repo/apollo13_blender.")
            return
        (WORKROOT / "apollo13_repo_settings.json").write_text(json.dumps({"url": REPO_URL, "branch": REPO_BRANCH, "project": str(PROJECT)}, indent=2))
        print(f"Ready. Using project: {PROJECT}")

clone_button.on_click(clone_selected_repository)


In [ ]:
# 4. Stage 1: validate configuration and runtime
!python3 scripts/validate_project.py
!$BLENDER --background --version
assert os.path.isfile('/usr/bin/ffmpeg') or subprocess.call(['which','ffmpeg']) == 0


In [ ]:
# 5. Stage 2: build every scene (headless), saves blends and build logs
!$BLENDER --background --python scripts/build_all.py
!$BLENDER --background --python scripts/build_assets.py
!python3 scripts/create_audio.py
!python3 scripts/validate_project.py --after-build


In [ ]:
# 6. Stage 3: asset/material contact sheet (1280x720 build asset scene)
!$BLENDER --background assets/generated/apollo13_assets.blend --python scripts/render_shot.py -- --shot S01_PAD_REVEAL --profile REVIEW --start 1 --end 1
from IPython.display import display, Image
print('Inspect generated REVIEW frame (asset contact-sheet proxy); production may replace with curated contact sheet.')


In [ ]:
# 7. Asset approval — intentionally false by default
APPROVE_ASSETS = False
assert APPROVE_ASSETS, 'Set APPROVE_ASSETS = True only after visual inspection.'


In [ ]:
# 8. Stage 4: representative stills (first/middle/final; six story-critical shots)
representative=['S03_IGNITION','S06_ODYSSEY_CALM','S09_VENTING','S10_CONTROL_ESTABLISH','S15_LUNAR_HORIZON','S26_CHUTES']
for shot in representative:
 !python3 scripts/render_all.py --profile REVIEW --shot {shot}
print('Review each shot folder; automated black/exposure checks should be recorded before approval.')


In [ ]:
# 9. Shot-test approval
APPROVE_SHOT_TESTS = False
assert APPROVE_SHOT_TESTS, 'Set True only after labeled still contact-sheet review.'


In [ ]:
# 10. Stage 5: short motion tests from complex shots, 854x480 intended override
# Render ~3 seconds each with a temporary PREVIEW override, then concatenate as previews/motion_test.mp4.
for shot in ['S03_IGNITION','S09_VENTING','S15_LUNAR_HORIZON','S24_PLASMA','S27_SPLASHDOWN']:
 !python3 scripts/render_all.py --profile PREVIEW --shot {shot}
!python3 scripts/assemble_video.py --profile PREVIEW --output previews/motion_test.mp4


In [ ]:
# 11. Motion-test approval
APPROVE_MOTION_TEST = False
assert APPROVE_MOTION_TEST, 'Set True after checking camera movement, flicker, clipping, and motion blur.'


In [ ]:
# 12. Stage 6: full exact-timing PREVIEW animatic
!python3 scripts/render_all.py --profile PREVIEW
!python3 scripts/assemble_video.py --profile PREVIEW --output previews/apollo13_full_animatic.mp4


In [ ]:
# 13. Animatic approval
APPROVE_FULL_ANIMATIC = False
assert APPROVE_FULL_ANIMATIC, 'Set True after timing/audio/title/missing-frame review.'


In [ ]:
# 14. Five FINAL-quality frames from demanding reentry shot
!python3 scripts/render_all.py --profile FINAL --shot S24_PLASMA --start 1585 --end 1589
# Retain `previews/final_quality_test/` for review; compare REVIEW and FINAL manually.


In [ ]:
# 15. Performance estimate
import json
m=json.load(open('production_manifest.json')); print(m['project_settings'], 'shots=',len(m['shots']))
print('Estimate remaining frames and disk from frames/FINAL before proceeding.')


In [ ]:
# 16. Final-quality approval
APPROVE_FINAL_QUALITY_TEST = False
assert APPROVE_FINAL_QUALITY_TEST, 'Set True only after final-quality performance/quality review.'


In [ ]:
# 17. Full FINAL rendering — deliberately blocked unless every approval is true
START_FINAL_RENDER = False
if not all([APPROVE_ASSETS, APPROVE_SHOT_TESTS, APPROVE_MOTION_TEST, APPROVE_FULL_ANIMATIC, APPROVE_FINAL_QUALITY_TEST, START_FINAL_RENDER]):
 raise RuntimeError('FINAL RENDER BLOCKED — Complete and approve all preview tests first.')
!python3 scripts/render_all.py --profile FINAL


In [ ]:
# 18. Missing-frame verification
!python3 scripts/validate_project.py --after-build
# assembly itself refuses missing frames


In [ ]:
# 19. Video assembly
!python3 scripts/assemble_video.py --profile FINAL


In [ ]:
# 20. Download and optional Drive backup
from google.colab import files
files.download('output/apollo13_the_long_way_home.mp4')
# !cp output/apollo13_the_long_way_home.mp4 /content/drive/MyDrive/apollo13_render/
